# DSS Analisis Kemiripan Logo Merek Dagang — **Run All**

Notebook ini disusun sebagai notebook deployment Colab yang idempotent dan dapat dijalankan dengan **Runtime → Run all**.

## Persiapan satu kali

1. Pastikan struktur Google Drive berikut tersedia:

   ```text
   MyDrive/project_skripsi/
   ├── App/
   │   ├── app.py
   │   └── ... file pendukung aplikasi lainnya
   ├── Models/
   │   ├── hybrid_best.pt
   │   ├── pdki_embeddings.npy
   │   ├── pdki_hsv.npy
   │   └── pdki_metadata.json
   └── Datasets/
       └── PDKI/
           ├── ... gambar logo
           └── ... CSV metadata opsional
   ```

2. Di Google Colab, buka panel **Secrets** (ikon kunci) lalu buat secret:
   - Nama: `NGROK_TOKEN`
   - Nilai: token ngrok Anda
   - Aktifkan akses notebook terhadap secret tersebut.

## Perbaikan utama

- Tidak menyimpan token ngrok di source code.
- Tidak melakukan patch teks terhadap `app.py`.
- Folder gambar dihubungkan menggunakan symbolic link `pdki_images`.
- Memvalidasi jumlah metadata, embedding, HSV, dan gambar sebelum aplikasi dijalankan.
- Memeriksa sintaks `app.py`.
- Menyimpan log Streamlit.
- Menunggu health endpoint Streamlit aktif sebelum membuat tunnel.
- Menggunakan konfigurasi keamanan Streamlit bawaan; XSRF tidak dinonaktifkan.
- Membuat runtime manifest untuk dokumentasi dan reproducibility.

In [1]:
# ============================================================
# CELL 1 — Konfigurasi
# ============================================================
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/project_skripsi")
SOURCE_APP_DIR = DRIVE_BASE / "App"
MODEL_DIR = DRIVE_BASE / "Models"
PDKI_DIR = DRIVE_BASE / "Datasets" / "PDKI"

APP_DIR = Path("/content/DSS_App")
PORT = 8501

# True: bangun ulang folder aplikasi lokal setiap Run all.
RESET_APP_DIR = True

# True: mencoba memperkaya pdki_metadata.json dari CSV di Datasets/PDKI.
ENRICH_METADATA = True

# Sebaiknya False agar file sumber di Drive tidak diubah tanpa sengaja.
WRITE_ENRICHED_METADATA_BACK_TO_DRIVE = False

# Batas waktu menunggu Streamlit siap.
HEALTH_TIMEOUT_SECONDS = 120

# Ekstensi gambar yang dihitung.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Konfigurasi siap.")
print("DRIVE_BASE:", DRIVE_BASE)
print("APP_DIR   :", APP_DIR)

Konfigurasi siap.
DRIVE_BASE: /content/drive/MyDrive/project_skripsi
APP_DIR   : /content/DSS_App


In [2]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Google Drive terhubung.")

Mounted at /content/drive
Google Drive terhubung.


In [3]:
# ============================================================
# CELL 3 — Instal dependency
# ============================================================
import subprocess
import sys

packages = [
    "streamlit>=1.40,<2",
    "pyngrok>=7,<8",
    "scikit-image>=0.22,<1",
    "grad-cam>=1.5,<2",
    "opencv-python-headless>=4.8,<5",
    "pandas>=2,<3",
    "Pillow>=10,<13",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *packages]
)

import cv2
import numpy as np
import pandas as pd
import PIL
import pyngrok
import skimage
import streamlit
import torch

print("Dependency siap:")
print("  Python    :", sys.version.split()[0])
print("  Streamlit :", streamlit.__version__)
print("  PyTorch   :", torch.__version__)
print("  NumPy     :", np.__version__)
print("  Pandas    :", pd.__version__)
print("  OpenCV    :", cv2.__version__)
print("  scikit-img:", skimage.__version__)
print("  Pillow    :", PIL.__version__)
print("  CUDA      :", torch.cuda.is_available())

Dependency siap:
  Python    : 3.12.13
  Streamlit : 1.60.0
  PyTorch   : 2.11.0+cpu
  NumPy     : 2.0.2
  Pandas    : 2.2.2
  OpenCV    : 4.14.0
  scikit-img: 0.25.2
  Pillow    : 11.3.0
  CUDA      : False


In [4]:
# ============================================================
# CELL 4 — Validasi sumber data dan artefak
# ============================================================
import hashlib
import json
import math
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

REQUIRED_FILES = {
    "app.py": SOURCE_APP_DIR / "app.py",
    "hybrid_best.pt": MODEL_DIR / "hybrid_best.pt",
    "pdki_embeddings.npy": MODEL_DIR / "pdki_embeddings.npy",
    "pdki_hsv.npy": MODEL_DIR / "pdki_hsv.npy",
    "pdki_metadata.json": MODEL_DIR / "pdki_metadata.json",
}

def human_size(num_bytes: int) -> str:
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

missing = [str(path) for path in REQUIRED_FILES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "File wajib berikut belum tersedia:\n- " + "\n- ".join(missing)
    )

if not PDKI_DIR.is_dir():
    raise FileNotFoundError(f"Folder dataset tidak ditemukan: {PDKI_DIR}")

image_paths = sorted(
    path
    for path in PDKI_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_paths:
    raise RuntimeError(f"Tidak ada gambar yang ditemukan di {PDKI_DIR}")

embeddings = np.load(REQUIRED_FILES["pdki_embeddings.npy"], mmap_mode="r")
hsv_features = np.load(REQUIRED_FILES["pdki_hsv.npy"], mmap_mode="r")

with REQUIRED_FILES["pdki_metadata.json"].open("r", encoding="utf-8") as handle:
    metadata = json.load(handle)

if not isinstance(metadata, list):
    raise TypeError("pdki_metadata.json harus berisi JSON list.")

problems = []

if embeddings.ndim != 2:
    problems.append(f"Embedding harus 2D, ditemukan shape {embeddings.shape}.")
if hsv_features.ndim != 2:
    problems.append(f"HSV harus 2D, ditemukan shape {hsv_features.shape}.")

counts = {
    "metadata": len(metadata),
    "embeddings": int(embeddings.shape[0]) if embeddings.ndim >= 1 else 0,
    "hsv": int(hsv_features.shape[0]) if hsv_features.ndim >= 1 else 0,
    "images": len(image_paths),
}

if len(set(counts.values())) != 1:
    problems.append(
        "Jumlah artefak tidak konsisten: "
        + ", ".join(f"{key}={value}" for key, value in counts.items())
    )

# Pemeriksaan finite dilakukan secara sampling agar hemat memori.
def finite_sample(array, max_rows=512) -> bool:
    if array.ndim != 2 or array.shape[0] == 0:
        return False
    rows = min(max_rows, array.shape[0])
    sample = np.asarray(array[:rows])
    return bool(np.isfinite(sample).all())

if not finite_sample(embeddings):
    problems.append("Sampel embedding mengandung NaN/Inf atau tidak valid.")
if not finite_sample(hsv_features):
    problems.append("Sampel fitur HSV mengandung NaN/Inf atau tidak valid.")

duplicate_filenames = sum(
    count - 1 for count in Counter(path.name for path in image_paths).values()
    if count > 1
)

summary = pd.DataFrame(
    [
        {
            "artefak": name,
            "lokasi": str(path),
            "ukuran": human_size(path.stat().st_size),
            "status": "OK",
        }
        for name, path in REQUIRED_FILES.items()
    ]
)

display(summary)

print("\nRingkasan konsistensi:")
for key, value in counts.items():
    print(f"  {key:10s}: {value:,}")
print("  embedding shape:", tuple(embeddings.shape))
print("  HSV shape      :", tuple(hsv_features.shape))
print("  nama file gambar duplikat:", duplicate_filenames)

if problems:
    raise RuntimeError(
        "Validasi artefak gagal:\n- " + "\n- ".join(problems)
    )

print("\nSemua artefak utama konsisten.")

,artefak,lokasi,ukuran,status
0,app.py,/content/drive/MyDrive/project_skripsi/App/app.py,86.2 KB,OK
1,hybrid_best.pt,/content/drive/MyDrive/project_skripsi/Models/...,107.0 MB,OK
2,pdki_embeddings.npy,/content/drive/MyDrive/project_skripsi/Models/...,6.5 MB,OK
3,pdki_hsv.npy,/content/drive/MyDrive/project_skripsi/Models/...,837.4 KB,OK
4,pdki_metadata.json,/content/drive/MyDrive/project_skripsi/Models/...,3.5 MB,OK



Ringkasan konsistensi:
  metadata  : 6,698
  embeddings: 6,698
  hsv       : 6,698
  images    : 6,698
  embedding shape: (6698, 256)
  HSV shape      : (6698, 32)
  nama file gambar duplikat: 6595

Semua artefak utama konsisten.


In [5]:
# ============================================================
# CELL 5 — Siapkan folder aplikasi lokal
# ============================================================
import os
import shutil
from pathlib import Path

if RESET_APP_DIR and APP_DIR.exists():
    shutil.rmtree(APP_DIR)

APP_DIR.mkdir(parents=True, exist_ok=True)

# Salin seluruh folder App agar assets/module pendukung ikut terbawa.
shutil.copytree(SOURCE_APP_DIR, APP_DIR, dirs_exist_ok=True)

# Salin artefak model ke root aplikasi, mengikuti struktur notebook lama.
for filename in (
    "hybrid_best.pt",
    "pdki_embeddings.npy",
    "pdki_hsv.npy",
    "pdki_metadata.json",
):
    source = MODEL_DIR / filename
    destination = APP_DIR / filename
    shutil.copy2(source, destination)
    print(f"OK copy: {filename} -> {destination}")

# app.py lama mengharapkan BASE_DIR/pdki_images.
# Gunakan symlink, bukan patch source code.
image_link = APP_DIR / "pdki_images"

if image_link.is_symlink() or image_link.exists():
    if image_link.is_symlink() or image_link.is_file():
        image_link.unlink()
    else:
        shutil.rmtree(image_link)

os.symlink(PDKI_DIR, image_link, target_is_directory=True)

if not image_link.exists():
    raise RuntimeError("Gagal membuat symbolic link folder gambar.")

print("\nFolder aplikasi lokal siap:", APP_DIR)
print("Symlink gambar:", image_link, "->", image_link.resolve())

OK copy: hybrid_best.pt -> /content/DSS_App/hybrid_best.pt
OK copy: pdki_embeddings.npy -> /content/DSS_App/pdki_embeddings.npy
OK copy: pdki_hsv.npy -> /content/DSS_App/pdki_hsv.npy
OK copy: pdki_metadata.json -> /content/DSS_App/pdki_metadata.json

Folder aplikasi lokal siap: /content/DSS_App
Symlink gambar: /content/DSS_App/pdki_images -> /content/drive/MyDrive/project_skripsi/Datasets/PDKI


In [6]:
# ============================================================
# CELL 6 — Perkaya metadata dari CSV (opsional dan aman)
# ============================================================
import json
import math
from collections import defaultdict
from pathlib import Path

import pandas as pd

LOCAL_METADATA_PATH = APP_DIR / "pdki_metadata.json"

def clean_value(value, max_length=None):
    if value is None:
        text = ""
    elif isinstance(value, float) and math.isnan(value):
        text = ""
    else:
        text = str(value).strip()

    if text.lower() in {"nan", "none", "null", "nat"}:
        text = ""

    if max_length is not None:
        text = text[:max_length]

    return text

def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin-1")

def metadata_folder_key(entry: dict):
    raw_path = clean_value(entry.get("path", "")).replace("\\", "/")
    if not raw_path:
        return ""
    parts = [part for part in raw_path.split("/") if part]
    try:
        index = parts.index("PDKI")
        return parts[index + 1] if index + 1 < len(parts) else ""
    except ValueError:
        return Path(raw_path).parent.name

if not ENRICH_METADATA:
    print("ENRICH_METADATA=False; tahap enrichment dilewati.")
else:
    csv_paths = sorted(
        path
        for path in PDKI_DIR.rglob("*.csv")
        if "batch_summary" not in path.name.lower()
    )

    if not csv_paths:
        print("Tidak ada CSV metadata; file JSON asli tetap digunakan.")
    else:
        key_rows = defaultdict(list)
        csv_errors = []

        for csv_path in csv_paths:
            try:
                frame = read_csv_robust(csv_path)
                filename_column = next(
                    (
                        column
                        for column in ("image_filename", "filename", "image")
                        if column in frame.columns
                    ),
                    None,
                )

                if filename_column is None:
                    print(f"SKIP {csv_path.name}: kolom nama file tidak ditemukan.")
                    continue

                folder_key = csv_path.stem

                for row in frame.to_dict(orient="records"):
                    filename = Path(clean_value(row.get(filename_column))).name
                    if filename:
                        key_rows[(folder_key, filename)].append(row)

            except Exception as exc:
                csv_errors.append((str(csv_path), repr(exc)))

        unique_lookup = {
            key: rows[0] for key, rows in key_rows.items() if len(rows) == 1
        }
        ambiguous_keys = {
            key for key, rows in key_rows.items() if len(rows) > 1
        }

        with LOCAL_METADATA_PATH.open("r", encoding="utf-8") as handle:
            local_metadata = json.load(handle)

        updated = 0
        unmatched = 0

        for entry in local_metadata:
            folder_key = metadata_folder_key(entry)
            filename = Path(clean_value(entry.get("filename", ""))).name
            key = (folder_key, filename)

            row = unique_lookup.get(key)
            if row is None:
                unmatched += 1
                continue

            nama_merek = clean_value(row.get("nama_merek"), 120)

            entry.update(
                {
                    "nama_merek": nama_merek,
                    "brand": nama_merek or clean_value(entry.get("brand"), 120),
                    "application_id": clean_value(row.get("application_id")),
                    "nomor_permohonan": clean_value(row.get("nomor_permohonan")),
                    "kelas_nice": clean_value(row.get("kelas_nice")),
                    "tahun": clean_value(row.get("tahun")),
                    "tanggal_permohonan": clean_value(
                        row.get("tanggal_permohonan")
                    ),
                    "status_permohonan": clean_value(
                        row.get("status_permohonan")
                    )
                    or "N/A",
                    "owner_name": clean_value(row.get("owner_name"), 120),
                    "class_desc": clean_value(row.get("class_desc"), 300),
                }
            )
            updated += 1

        with LOCAL_METADATA_PATH.open("w", encoding="utf-8") as handle:
            json.dump(
                local_metadata,
                handle,
                indent=2,
                ensure_ascii=False,
            )

        if WRITE_ENRICHED_METADATA_BACK_TO_DRIVE:
            shutil.copy2(
                LOCAL_METADATA_PATH,
                MODEL_DIR / "pdki_metadata.json",
            )

        print("Metadata enrichment selesai:")
        print(f"  CSV dibaca         : {len(csv_paths)}")
        print(f"  entri diperbarui   : {updated:,}")
        print(f"  entri tidak cocok  : {unmatched:,}")
        print(f"  key ambigu         : {len(ambiguous_keys):,}")
        print(f"  error CSV          : {len(csv_errors):,}")

        if csv_errors:
            print("\nContoh error CSV:")
            for path, error in csv_errors[:10]:
                print(" -", path, "=>", error)

Metadata enrichment selesai:
  CSV dibaca         : 135
  entri diperbarui   : 6,698
  entri tidak cocok  : 0
  key ambigu         : 0
  error CSV          : 0


In [7]:
# ============================================================
# CELL 7 — Smoke test, validasi sintaks, dan runtime manifest
# ============================================================
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone

import numpy as np
import streamlit
import torch

APP_FILE = APP_DIR / "app.py"

syntax_test = subprocess.run(
    [sys.executable, "-m", "py_compile", str(APP_FILE)],
    capture_output=True,
    text=True,
)

if syntax_test.returncode != 0:
    raise SyntaxError(
        "app.py gagal dikompilasi:\n"
        + (syntax_test.stderr or syntax_test.stdout)
    )

local_embeddings = np.load(APP_DIR / "pdki_embeddings.npy", mmap_mode="r")
local_hsv = np.load(APP_DIR / "pdki_hsv.npy", mmap_mode="r")

with (APP_DIR / "pdki_metadata.json").open("r", encoding="utf-8") as handle:
    local_metadata = json.load(handle)

if not (
    len(local_metadata)
    == local_embeddings.shape[0]
    == local_hsv.shape[0]
):
    raise RuntimeError(
        "Artefak lokal tidak konsisten setelah proses penyalinan/enrichment."
    )

manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "streamlit": streamlit.__version__,
    "cuda_available": torch.cuda.is_available(),
    "app_sha256": sha256_file(APP_FILE),
    "model_sha256": sha256_file(APP_DIR / "hybrid_best.pt"),
    "embedding_shape": list(local_embeddings.shape),
    "hsv_shape": list(local_hsv.shape),
    "metadata_count": len(local_metadata),
    "image_count": len(image_paths),
    "drive_base": str(DRIVE_BASE),
    "app_dir": str(APP_DIR),
}

manifest_path = APP_DIR / "runtime_manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

print("Smoke test berhasil.")
print("Manifest:", manifest_path)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

Smoke test berhasil.
Manifest: /content/DSS_App/runtime_manifest.json
{
  "generated_at_utc": "2026-07-31T14:55:29.155339+00:00",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cpu",
  "streamlit": "1.60.0",
  "cuda_available": false,
  "app_sha256": "f127dbffaff19032346ff3e6fd3a07c6cec6af86d5b4b662f9c00475afaa218e",
  "model_sha256": "3b55ee0609063f2db22efec1b242f0c6c33110148b811bde0117ec08978346cd",
  "embedding_shape": [
    6698,
    256
  ],
  "hsv_shape": [
    6698,
    32
  ],
  "metadata_count": 6698,
  "image_count": 6698,
  "drive_base": "/content/drive/MyDrive/project_skripsi",
  "app_dir": "/content/DSS_App"
}


In [8]:
# ============================================================
# CELL 8 — Ambil NGROK_TOKEN dari Colab Secrets
# ============================================================
import os

NGROK_TOKEN = os.getenv("NGROK_TOKEN", "3G5dWA2BCHpYeHmkFiIUkmngZU0_tx9wM8gxpvesZ3SWVLfA").strip()

if not NGROK_TOKEN:
    try:
        from google.colab import userdata

        NGROK_TOKEN = (userdata.get("NGROK_TOKEN") or "").strip()
    except Exception:
        NGROK_TOKEN = ""

if not NGROK_TOKEN:
    raise RuntimeError(
        "NGROK_TOKEN belum tersedia. Tambahkan secret bernama "
        "'NGROK_TOKEN' melalui panel Secrets di Google Colab, "
        "aktifkan akses notebook, lalu jalankan kembali Run all."
    )

print("NGROK_TOKEN berhasil dibaca dari secret/environment.")

NGROK_TOKEN berhasil dibaca dari secret/environment.


In [9]:
# ============================================================
# CELL 9 — Jalankan Streamlit
# ============================================================

import os
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

from pyngrok import conf, ngrok

PID_FILE = APP_DIR / "streamlit.pid"
LOG_FILE = APP_DIR / "streamlit.log"

def process_is_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def stop_previous_streamlit():
    if not PID_FILE.exists():
        return

    try:
        pid = int(PID_FILE.read_text(encoding="utf-8").strip())
    except Exception:
        PID_FILE.unlink(missing_ok=True)
        return

    if process_is_alive(pid):
        print(f"Menghentikan proses Streamlit lama: PID {pid}")
        try:
            os.kill(pid, signal.SIGTERM)
            for _ in range(20):
                if not process_is_alive(pid):
                    break
                time.sleep(0.25)
        except OSError:
            pass

    PID_FILE.unlink(missing_ok=True)

def tail_text(path: Path, max_lines: int = 120) -> str:
    if not path.exists():
        return "(log belum tersedia)"
    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()
    return "\n".join(lines[-max_lines:])

stop_previous_streamlit()
ngrok.kill()
# Add a small delay after killing ngrok processes to allow the service to update its state.
time.sleep(2) # Added this line

command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_DIR / "app.py"),
    "--server.address",
    "0.0.0.0",
    "--server.port",
    str(PORT),
    "--server.headless",
    "true",
    "--browser.gatherUsageStats",
    "false",
]

environment = os.environ.copy()
environment.update(
    {
        "DSS_APP_DIR": str(APP_DIR),
        "DSS_PDKI_DIR": str(PDKI_DIR),
        "DSS_MODEL_DIR": str(APP_DIR),
        "PYTHONUNBUFFERED": "1",
    }
)

log_handle = LOG_FILE.open("w", encoding="utf-8")

STREAMLIT_PROCESS = subprocess.Popen(
    command,
    cwd=str(APP_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=environment,
)

PID_FILE.write_text(str(STREAMLIT_PROCESS.pid), encoding="utf-8")

health_url = f"http://127.0.0.1:{PORT}/_stcore/health"
deadline = time.time() + HEALTH_TIMEOUT_SECONDS
ready = False
last_error = None

while time.time() < deadline:
    if STREAMLIT_PROCESS.poll() is not None:
        break

    try:
        with urllib.request.urlopen(health_url, timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception as exc:
        last_error = repr(exc)
        time.sleep(1)

if not ready:
    log_handle.flush()
    raise RuntimeError(
        "Streamlit gagal mencapai status sehat.\n"
        f"Proses return code: {STREAMLIT_PROCESS.poll()}\n"
        f"Health error terakhir: {last_error}\n\n"
        "Log terakhir:\n"
        + tail_text(LOG_FILE)
    )

conf.get_default().auth_token = NGROK_TOKEN
NGROK_TUNNEL = ngrok.connect(addr=PORT, bind_tls=True)

print("\n" + "=" * 72)
print("DSS ANALISIS KEMIRIPAN LOGO SIAP DIAKSES")
print("=" * 72)
print("URL lokal :", f"http://127.0.0.1:{PORT}")
print("URL publik:", NGROK_TUNNEL.public_url)
print("PID       :", STREAMLIT_PROCESS.pid)
print("Log       :", LOG_FILE)
print("=" * 72)



DSS ANALISIS KEMIRIPAN LOGO SIAP DIAKSES
URL lokal : http://127.0.0.1:8501
URL publik: https://broadness-lurch-deletion.ngrok-free.dev
PID       : 2295
Log       : /content/DSS_App/streamlit.log


In [10]:
# ============================================================
# CELL 10 — Status akhir dan log ringkas
# ============================================================
import time
import urllib.request

time.sleep(2)

status = "berjalan" if STREAMLIT_PROCESS.poll() is None else "berhenti"
print("Status proses Streamlit:", status)
print("Public URL:", NGROK_TUNNEL.public_url)

try:
    with urllib.request.urlopen(
        f"http://127.0.0.1:{PORT}/_stcore/health",
        timeout=5,
    ) as response:
        print("Health endpoint:", response.status, response.read().decode())
except Exception as exc:
    print("Health endpoint error:", repr(exc))

print("\n--- 80 baris log terakhir ---")
print(tail_text(LOG_FILE, max_lines=80))

Status proses Streamlit: berjalan
Public URL: https://broadness-lurch-deletion.ngrok-free.dev
Health endpoint: 200 ok

--- 80 baris log terakhir ---
2026-07-31 14:55:32.619 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.252.180:8501



In [11]:
# ═══════════════════════════════════════════════
# Update app.py & Restart
# Jalankan setelah edit app.py di Drive
# ═══════════════════════════════════════════════
import shutil, subprocess, time
from pyngrok import ngrok, conf

DRIVE_BASE  = '/content/drive/MyDrive/project_skripsi'
APP_DIR     = '/content/DSS_App'
PDKI_DIR    = f'{DRIVE_BASE}/Datasets/PDKI'
NGROK_TOKEN = '3G5dWA2BCHpYeHmkFiIUkmngZU0_tx9wM8gxpvesZ3SWVLfA'

shutil.copy2(f'{DRIVE_BASE}/App/app.py', f'{APP_DIR}/app.py')

with open(f'{APP_DIR}/app.py', 'r', encoding='utf-8') as f:
    c = f.read()
c = c.replace('IMAGES_DIR    = BASE_DIR / "pdki_images"',
              f'IMAGES_DIR    = Path("{PDKI_DIR}")')
OLD = ('    final_scores  = alpha * hybrid_scores + (1 - alpha) * color_scores\n'
       '    top_idx       = np.argsort(final_scores)[::-1][:k]')
NEW = ('    final_scores  = alpha * hybrid_scores + (1 - alpha) * color_scores\n'
       '    if final_scores.max() > 0.9999:\n'
       '        final_scores[final_scores.argmax()] = -1\n'
       '    top_idx       = np.argsort(final_scores)[::-1][:k]')
if OLD in c: c = c.replace(OLD, NEW)
with open(f'{APP_DIR}/app.py', 'w', encoding='utf-8') as f:
    f.write(c)
print('OK app.py diupdate dan di-patch')

!pkill -f streamlit 2>/dev/null || true
ngrok.kill()
time.sleep(3)

conf.get_default().auth_token = NGROK_TOKEN
subprocess.Popen(
    ['streamlit', 'run', f'{APP_DIR}/app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(8)
tunnel = ngrok.connect(8501)
print(f'App di-restart. URL: {tunnel.public_url}')

OK app.py diupdate dan di-patch
^C
App di-restart. URL: https://broadness-lurch-deletion.ngrok-free.dev
